# Notebook 9: Support Vector Machines (SVM) – Theory & Kernel Trick
### Part 9/30 – ML Mastery Series for Python Experts

## What Makes SVM Special – Maximum Margin Intuition

Support Vector Machines aren't just another classifier—they're a fundamentally different way of thinking about decision boundaries:

- **Finds the widest street between classes**: Instead of fitting a line that merely separates, SVM finds the hyperplane that maximizes the margin—the perpendicular distance to the nearest data points from each class
- **Robust to outliers in some cases**: By focusing only on the boundary region, SVM can ignore distant outliers that might skew other models
- **Focuses only on support vectors**: The model is entirely defined by a subset of training points (the support vectors), making prediction dependent only on these critical examples
- **Memory efficient for many datasets**: Once trained, only support vectors need to be stored, not the entire training set
- **But sensitive to scaling and hyperparameters**: Feature scales dramatically affect margin calculations, and the kernel/C/gamma choices can make or break performance
- **The kernel trick is pure mathematical elegance**: Map to infinite dimensions without computing the coordinates explicitly—this is what separates SVM from linear models
- **Geometric interpretation beats probabilistic**: While logistic regression gives probabilities, SVM gives you a geometric "buffer zone" with clear geometric meaning

## Learning Objectives

By the end of this notebook, you will:

- **Understand hard vs soft margin**: Know when data is linearly separable and when to allow violations
- **Interpret support vectors**: Identify which points actually define your decision boundary
- **Visualize margin & decision boundary**: Plot the "street" and see how wide or narrow it becomes
- **Use kernel trick to handle non-linear data**: Transform problems without explicit feature engineering
- **Compare linear/RBF/polynomial kernels**: Know when each shines and when they fail
- **Tune C and gamma effectively**: Navigate the bias-variance trade-off for SVM specifically
- **Handle multiclass with OvO/OvR**: Understand the computational implications of each strategy
- **Apply scaling religiously**: See why StandardScaler is non-negotiable for SVM
- **Debug SVM failures**: Recognize overfitting from high gamma or memorization from high C

## ⚡ 1. Hard-Margin SVM – Linearly Separable Data

We start with the ideal case: perfectly linearly separable data. The hard-margin SVM finds the hyperplane that separates classes with zero errors and maximum margin.

Mathematically, we solve:
$$\min_{w,b} \frac{1}{2}||w||^2 \quad \text{subject to} \quad y_i(w^T x_i + b) \geq 1 \quad \forall i$$

This is a quadratic programming problem with linear constraints.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.svm import SVC
import seaborn as sns

# Set style for professional plots
sns.set_theme()
%matplotlib inline

# Generate perfectly separable 2D data
X, y = make_classification(
    n_samples=200,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    class_sep=2.0,  # Large separation ensures linear separability
    random_state=42
)

# Hard-margin approximation: very large C (effectively infinite penalty for violations)
svm_hard = SVC(kernel='linear', C=1e10, random_state=42)
svm_hard.fit(X, y)

print(f"Number of support vectors: {svm_hard.n_support_}")
print(f"Total support vectors: {svm_hard.support_.shape[0]}")
print(f"Margin width (approx): {2 / np.linalg.norm(svm_hard.coef_)}")

Number of support vectors: [5 4]
Total support vectors: 9
Margin width (approx): 0.00029967549446305227


In [ ]:
def plot_svm_boundary(X, y, model, title, ax=None):
    """Plot SVM decision boundary with margins and support vectors."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))
    
    # Create mesh for decision surface
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    
    # Get decision function values
    Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot decision regions
    ax.contourf(xx, yy, Z, levels=[-1, 0, 1], alpha=0.3, 
                colors=['blue', 'white', 'red'], extend='both')
    
    # Plot decision boundary and margins
    ax.contour(xx, yy, Z, levels=[-1, 0, 1], colors='k', 
               linestyles=['--', '-', '--'], linewidths=1.5)
    
    # Plot data points
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', 
                        edgecolors='k', s=50, zorder=5)
    
    # Highlight support vectors
    ax.scatter(X[model.support_, 0], X[model.support_, 1], 
              s=200, facecolors='none', edgecolors='lime', 
              linewidths=2, label='Support Vectors', zorder=6)
    
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.set_title(title)
    ax.legend()
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    
    return ax

# Visualize hard-margin SVM
fig, ax = plt.subplots(figsize=(10, 6))
plot_svm_boundary(X, y, svm_hard, 
                  'Hard-Margin SVM (C=1e10)\nZero violations, maximum margin', ax)
plt.tight_layout()
plt.show()

print("Notice: Only 2-3 points per class define the entire boundary (support vectors).")

## 🛡️ 2. Soft-Margin SVM – Handling Non-Separable Data

Real data is rarely perfectly separable. The soft-margin SVM introduces slack variables $\xi_i$ and parameter $C$ to balance margin width against classification errors:

$$\min_{w,b,\xi} \frac{1}{2}||w||^2 + C\sum_{i=1}^n \xi_i$$

- **Small C**: Wider margin, more violations allowed (smoother boundary, higher bias)
- **Large C**: Narrow margin, fewer violations (complex boundary, higher variance)

This is the hinge loss formulation: $\max(0, 1 - y_i(w^T x_i + b))$

In [ ]:
from sklearn.datasets import make_moons

# Generate non-linearly separable data (moons with noise)
X_moons, y_moons = make_moons(n_samples=300, noise=0.25, random_state=42)

# Try to fit linear SVM with different C values
C_values = [0.01, 0.1, 1, 10, 100]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, C in enumerate(C_values):
    svm_linear = SVC(kernel='linear', C=C, random_state=42)
    svm_linear.fit(X_moons, y_moons)
    
    plot_svm_boundary(X_moons, y_moons, svm_linear, 
                     f'Linear SVM (C={C})\nSupport vectors: {svm_linear.support_.shape[0]}', 
                     ax=axes[idx])
    
    # Calculate accuracy
    acc = svm_linear.score(X_moons, y_moons)
    axes[idx].text(0.05, 0.95, f'Accuracy: {acc:.3f}', 
                   transform=axes[idx].transAxes, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Remove empty subplot
axes[5].axis('off')
plt.tight_layout()
plt.show()

print("Observation: Small C allows margin violations (points inside margin), large C forces correct classification.")

In [ ]:
# Quantify the trade-off: margin width vs training accuracy
results = []
for C in np.logspace(-2, 3, 20):
    svm = SVC(kernel='linear', C=C, random_state=42)
    svm.fit(X_moons, y_moons)
    
    # Approximate margin width from dual coefficients
    margin_width = 2 / np.linalg.norm(svm.coef_) if np.linalg.norm(svm.coef_) > 0 else 0
    
    results.append({
        'C': C,
        'accuracy': svm.score(X_moons, y_moons),
        'n_support_vectors': svm.support_.shape[0],
        'margin_width': margin_width
    })

import pandas as pd
results_df = pd.DataFrame(results)

# Plot trade-off
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.semilogx(results_df['C'], results_df['accuracy'], 'b-o', label='Accuracy')
ax1.set_xlabel('C (regularization parameter)')
ax1.set_ylabel('Training Accuracy')
ax1.set_title('Accuracy vs C')
ax1.grid(True, alpha=0.3)

ax2.semilogx(results_df['C'], results_df['margin_width'], 'r-s', label='Margin Width')
ax2.set_xlabel('C (regularization parameter)')
ax2.set_ylabel('Approximate Margin Width')
ax2.set_title('Margin Width vs C')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key insight: As C increases, margin shrinks and accuracy increases (potentially overfitting).")

## 🌀 3. The Kernel Trick – Going Non-Linear

The kernel trick is SVM's superpower. Instead of explicitly mapping features to high-dimensional space $\phi(x)$, we compute similarities directly using a kernel function $K(x_i, x_j) = \phi(x_i)^T \phi(x_j)$.

Common kernels:
- **Linear**: $K(x_i, x_j) = x_i^T x_j$ (original space)
- **Polynomial**: $K(x_i, x_j) = (\gamma x_i^T x_j + r)^d$ (implicit feature expansion)
- **RBF (Gaussian)**: $K(x_i, x_j) = \exp(-\gamma ||x_i - x_j||^2)$ (infinite-dimensional mapping)

The decision function becomes: $f(x) = \sum_{i \in SV} \alpha_i y_i K(x_i, x) + b$

In [ ]:
# Compare different kernels on the moons dataset
kernels = [
    ('linear', {'kernel': 'linear', 'C': 1}),
    ('poly (degree=3)', {'kernel': 'poly', 'degree': 3, 'C': 1, 'gamma': 'scale'}),
    ('rbf (gamma=0.5)', {'kernel': 'rbf', 'C': 1, 'gamma': 0.5}),
    ('rbf (gamma=2)', {'kernel': 'rbf', 'C': 1, 'gamma': 2})
]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for idx, (name, params) in enumerate(kernels):
    svm = SVC(**params, random_state=42)
    svm.fit(X_moons, y_moons)
    
    # Create fine mesh for smooth boundaries
    x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
    y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot decision regions
    axes[idx].contourf(xx, yy, Z, alpha=0.4, cmap='coolwarm')
    
    # Plot decision boundary
    Z_dec = svm.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    axes[idx].contour(xx, yy, Z_dec, levels=[0], colors='k', linewidths=2)
    
    # Plot data
    axes[idx].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, 
                     cmap='coolwarm', edgecolors='k', s=50)
    
    # Highlight support vectors
    axes[idx].scatter(X_moons[svm.support_, 0], X_moons[svm.support_, 1],
                     s=150, facecolors='none', edgecolors='lime', linewidths=2)
    
    acc = svm.score(X_moons, y_moons)
    axes[idx].set_title(f'{name}\nAccuracy: {acc:.3f}, SVs: {svm.support_.shape[0]}')
    axes[idx].set_xlim(x_min, x_max)
    axes[idx].set_ylim(y_min, y_max)

plt.suptitle('Kernel Comparison: Linear vs Polynomial vs RBF', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print("Linear fails on non-linear data. Polynomial captures some curvature. RBF adapts locally.")

## 🎯 4. RBF Kernel in Depth – Gamma & C Trade-off

The RBF kernel has two critical hyperparameters:
- **C**: Regularization (as before)
- **gamma ($\gamma$)**: Inverse of the radius of influence of support vectors
  - High gamma: Each SV has small influence → complex, wiggly boundary (overfitting)
  - Low gamma: Each SV has large influence → smooth boundary (underfitting)

Mathematically: $\gamma = \frac{1}{2\sigma^2}$ where $\sigma$ is the Gaussian width

In [ ]:
from sklearn.datasets import make_circles

# Create concentric circles (definitely non-linear)
X_circles, y_circles = make_circles(n_samples=400, factor=0.3, noise=0.1, random_state=42)

# Grid of gamma and C values
gamma_values = [0.1, 1, 10, 100]
C_values = [0.1, 1, 10, 100]

fig, axes = plt.subplots(len(gamma_values), len(C_values), figsize=(16, 16))

for i, gamma in enumerate(gamma_values):
    for j, C in enumerate(C_values):
        svm = SVC(kernel='rbf', C=C, gamma=gamma, random_state=42)
        svm.fit(X_circles, y_circles)
        
        # Create mesh
        x_min, x_max = X_circles[:, 0].min() - 0.2, X_circles[:, 0].max() + 0.2
        y_min, y_max = X_circles[:, 1].min() - 0.2, X_circles[:, 1].max() + 0.2
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                             np.linspace(y_min, y_max, 200))
        
        Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        
        # Plot
        axes[i, j].contourf(xx, yy, Z, alpha=0.4, cmap='coolwarm')
        axes[i, j].scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles,
                          cmap='coolwarm', edgecolors='k', s=30, alpha=0.7)
        
        acc = svm.score(X_circles, y_circles)
        axes[i, j].set_title(f'γ={gamma}, C={C}\nAcc: {acc:.3f}', fontsize=9)
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

# Add row and column labels
for i, gamma in enumerate(gamma_values):
    axes[i, 0].set_ylabel(f'γ={gamma}', rotation=0, size='large', labelpad=30)
for j, C in enumerate(C_values):
    axes[0, j].set_title(f'C={C}\n' + axes[0, j].get_title(), fontsize=9)

plt.suptitle('RBF Kernel: Gamma (rows) vs C (columns) Trade-off', fontsize=16, y=0.92)
plt.tight_layout()
plt.show()

print("Top-left: Underfitting (smooth boundary misses pattern)")
print("Bottom-right: Overfitting (boundary wiggles around every point)")

In [ ]:
# Automated search for best combination
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.01, 0.1, 1, 10]
}

grid_search = GridSearchCV(
    SVC(kernel='rbf', random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_circles, y_circles)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.4f}")

# Visualize best model
best_svm = grid_search.best_estimator_
fig, ax = plt.subplots(figsize=(8, 6))

xx, yy = np.meshgrid(np.linspace(X_circles[:, 0].min() - 0.2, X_circles[:, 0].max() + 0.2, 200),
                     np.linspace(X_circles[:, 1].min() - 0.2, X_circles[:, 1].max() + 0.2, 200))
Z = best_svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

ax.contourf(xx, yy, Z, alpha=0.4, cmap='coolwarm')
ax.scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap='coolwarm', edgecolors='k')
ax.set_title(f'Optimal RBF SVM\nC={grid_search.best_params_["C"]}, γ={grid_search.best_params_["gamma"]}')
plt.show()

## 🔀 5. Multiclass SVM – OvO vs OvR

SVM is inherently binary. For multiclass problems, sklearn implements two strategies:

- **One-vs-One (OvO)**: Train $\binom{k}{2}$ classifiers (one per pair), predict by voting
  - More models but each trains on subset (faster for small datasets)
  - Default in sklearn's SVC
  
- **One-vs-Rest (OvR)**: Train $k$ classifiers (one per class vs all others)
  - Fewer models but each trains on full dataset
  - More memory efficient for many classes

The `decision_function_shape` parameter controls this.

In [8]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

# Load multiclass dataset
wine = load_wine()
X_wine, y_wine = wine.data, wine.target

# Split and scale (CRITICAL for SVM)
X_train, X_test, y_train, y_test = train_test_split(
    X_wine, y_wine, test_size=0.3, random_state=42, stratify=y_wine
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Classes: {wine.target_names}")
print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

Classes: ['class_0' 'class_1' 'class_2']
Training samples: 124, Test samples: 54


In [ ]:
# Compare OvO vs OvR
strategies = ['ovo', 'ovr']
results = {}

for strategy in strategies:
    svm = SVC(kernel='rbf', C=1, gamma='scale', 
              decision_function_shape=strategy, random_state=42)
    svm.fit(X_train_scaled, y_train)
    
    y_pred = svm.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    
    results[strategy] = {
        'model': svm,
        'accuracy': acc,
        'n_support_vectors': svm.support_.shape[0],
        'support_vectors_per_class': svm.n_support_
    }
    
    print(f"\n{strategy.upper()} Strategy:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  Total support vectors: {svm.support_.shape[0]}")
    print(f"  Support vectors per class: {svm.n_support_}")

# Visualize confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, strategy in enumerate(strategies):
    y_pred = results[strategy]['model'].predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=wine.target_names, yticklabels=wine.target_names)
    axes[idx].set_title(f'{strategy.upper()} Strategy\nAccuracy: {results[strategy]["accuracy"]:.3f}')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('True')

plt.tight_layout()
plt.show()

print(f"\nNote: OvO trains {3*(3-1)//2}=3 classifiers, OvR trains 3 classifiers for this 3-class problem.")

## ⚙️ 6. Practical Tips – Scaling & Tuning

SVM is notoriously sensitive to feature scaling. Here's why and how to fix it, plus a proper tuning pipeline.

In [10]:
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

# Load breast cancer dataset (features have vastly different scales)
cancer = load_breast_cancer()
X_cancer, y_cancer = cancer.data, cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=42, stratify=y_cancer
)

print("Feature scale comparison (first 5 features):")
print(f"Means: {X_train[:, :5].mean(axis=0)}")
print(f"Stds:  {X_train[:, :5].std(axis=0)}")
print("\nNotice: Some features 1000x larger than others. SVM will ignore small-scale features!")

Feature scale comparison (first 5 features):
Means: [1.40672132e+01 1.92473626e+01 9.15574066e+01 6.48541099e+02
 9.61674286e-02]
Stds:  [3.49553212e+00 4.40044714e+00 2.41226787e+01 3.44565295e+02
 1.34429172e-02]

Notice: Some features 1000x larger than others. SVM will ignore small-scale features!


In [11]:
# Demonstrate scaling impact
svm_unscaled = SVC(kernel='rbf', C=1, gamma='scale', random_state=42)
svm_scaled = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=1, gamma='scale', random_state=42))
])

# Cross-validation scores
cv_unscaled = cross_val_score(svm_unscaled, X_train, y_train, cv=5, scoring='accuracy')
cv_scaled = cross_val_score(svm_scaled, X_train, y_train, cv=5, scoring='accuracy')

print(f"Unscaled SVM CV accuracy: {cv_unscaled.mean():.4f} (+/- {cv_unscaled.std():.4f})")
print(f"Scaled SVM CV accuracy:   {cv_scaled.mean():.4f} (+/- {cv_scaled.std():.4f})")
print(f"\nImprovement from scaling: {(cv_scaled.mean() - cv_unscaled.mean()):.4f}")

# Fit on full training set and test
svm_unscaled.fit(X_train, y_train)
svm_scaled.fit(X_train, y_train)

print(f"\nTest accuracy unscaled: {svm_unscaled.score(X_test, y_test):.4f}")
print(f"Test accuracy scaled:   {svm_scaled.score(X_test, y_test):.4f}")

Unscaled SVM CV accuracy: 0.9099 (+/- 0.0290)
Scaled SVM CV accuracy:   0.9714 (+/- 0.0179)

Improvement from scaling: 0.0615

Test accuracy unscaled: 0.9298
Test accuracy scaled:   0.9825


In [12]:
# Proper hyperparameter tuning with Pipeline
from sklearn.model_selection import StratifiedKFold

# Define pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', random_state=42))
])

# Parameter grid (note: use 'svm__' prefix for pipeline steps)
param_grid = {
    'svm__C': np.logspace(-2, 3, 6),      # 0.01 to 1000
    'svm__gamma': np.logspace(-3, 2, 6)   # 0.001 to 100
}

# Grid search with cross-validation
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.4f}")
print(f"Test set accuracy: {grid_search.score(X_test, y_test):.4f}")

# Show top 5 configurations
results_df = pd.DataFrame(grid_search.cv_results_)
top5 = results_df.nlargest(5, 'mean_test_score')[['param_svm__C', 'param_svm__gamma', 'mean_test_score', 'std_test_score']]
print("\nTop 5 configurations:")
print(top5.to_string(index=False))

Fitting 5 folds for each of 36 candidates, totalling 180 fits

Best parameters: {'svm__C': np.float64(10.0), 'svm__gamma': np.float64(0.01)}
Best CV accuracy: 0.9758
Test set accuracy: 0.9825

Top 5 configurations:
 param_svm__C  param_svm__gamma  mean_test_score  std_test_score
         10.0             0.010         0.975824        0.016150
          1.0             0.010         0.971429        0.014906
         10.0             0.001         0.971429        0.020382
        100.0             0.001         0.971429        0.011207
       1000.0             0.001         0.969231        0.012815


## 📊 7. Comparing SVM to Previous Models

Let's benchmark SVM against the models from previous notebooks on the same dataset.

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate
import time

# Use wine dataset for comparison
wine = load_wine()
X, y = wine.data, wine.target

# Define models
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (Linear)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='linear', random_state=42))
    ]),
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', random_state=42))
    ])
}

# Cross-validate with timing
comparison_results = []

for name, model in models.items():
    start_time = time.time()
    
    scores = cross_validate(
        model, X, y,
        cv=5,
        scoring=['accuracy', 'f1_weighted'],
        return_train_score=True,
        n_jobs=-1
    )
    
    elapsed = time.time() - start_time
    
    comparison_results.append({
        'Model': name,
        'Test Accuracy': scores['test_accuracy'].mean(),
        'Test F1': scores['test_f1_weighted'].mean(),
        'Train Accuracy': scores['train_accuracy'].mean(),
        'Fit Time (s)': scores['fit_time'].sum(),
        'Total Time (s)': elapsed
    })

comparison_df = pd.DataFrame(comparison_results)
print(comparison_df.round(4).to_string(index=False))

              Model  Test Accuracy  Test F1  Train Accuracy  Fit Time (s)  Total Time (s)
Logistic Regression         0.9832   0.9832          1.0000        0.1202          0.1158
      Decision Tree         0.8654   0.8633          1.0000        0.0309          0.2312
      Random Forest         0.9721   0.9719          1.0000        3.3611          1.4838
       SVM (Linear)         0.9606   0.9606          1.0000        0.0270          0.0487
          SVM (RBF)         0.9833   0.9835          0.9986        0.0258          0.0565


In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Accuracy comparison
axes[0].barh(comparison_df['Model'], comparison_df['Test Accuracy'], color='skyblue')
axes[0].set_xlabel('Test Accuracy')
axes[0].set_title('Cross-Validation Accuracy')
axes[0].set_xlim(0.8, 1.0)
for i, v in enumerate(comparison_df['Test Accuracy']):
    axes[0].text(v + 0.005, i, f'{v:.3f}', va='center')

# Overfitting check (train vs test)
x_pos = np.arange(len(comparison_df))
width = 0.35
axes[1].bar(x_pos - width/2, comparison_df['Train Accuracy'], width, label='Train', alpha=0.8)
axes[1].bar(x_pos + width/2, comparison_df['Test Accuracy'], width, label='Test', alpha=0.8)
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Train vs Test Accuracy\n(Gap indicates overfitting)')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(comparison_df['Model'], rotation=45, ha='right')
axes[1].legend()
axes[1].set_ylim(0.8, 1.05)

# Training time
axes[2].barh(comparison_df['Model'], comparison_df['Fit Time (s)'], color='coral')
axes[2].set_xlabel('Total Fit Time (s)')
axes[2].set_title('Training Time (5-fold CV)')
axes[2].set_xscale('log')

plt.tight_layout()
plt.show()

print("\nKey observations:")
print("- SVM RBF often achieves highest accuracy on small-medium datasets")
print("- Random Forest is competitive and often faster")
print("- Decision Tree overfits most (largest train-test gap)")
print("- Linear SVM is fastest among SVM variants")

## ⚠️ Common Pitfalls & Pro Tips

**Critical Mistakes to Avoid:**

- **Forgetting to scale features** → Disastrous for SVM. Always use StandardScaler or MinMaxScaler. Unscaled features make the kernel distance computations meaningless.
- **Wrong gamma scale** → Gamma too high = overfitting (memorizes training points). Gamma too low = underfitting (linear-like behavior). Start with 'scale' or 'auto' and tune log-uniformly.
- **Large C on noisy data** → Forces SVM to memorize noise. If your data has label noise, prefer smaller C for smoother boundaries.
- **Ignoring class imbalance** → SVM optimizes accuracy, not balanced accuracy. Use `class_weight='balanced'` for imbalanced datasets.
- **Multiclass OvO memory explosion** → OvO trains $O(k^2)$ models. For $k>10$ classes, OvR is usually more memory efficient.
- **SVM on very large datasets** → $O(n^2)$ to $O(n^3)$ complexity. For $n>10000$, prefer LinearSVC (approximate) or other models.
- **Probability=False by default** → SVM doesn't naturally output probabilities. Use `probability=True` to enable Platt scaling (sigmoid fit), but this slows training significantly.
- **Not checking support vector ratio** → If >50% of data are SVs, your model is likely overfitting or misconfigured.
- **Using RBF when linear suffices** → Always try linear first (faster, more interpretable). Use RBF only when linear fails.
- **Grid search without stratification** → Always use StratifiedKFold for classification to maintain class proportions in folds.

## 📝 Exercises

Test your understanding with these progressively challenging exercises:

**Easy:**
1. Fit a linear SVM on the Iris dataset (binary: Setosa vs. rest). Plot the decision boundary, margin lines, and highlight support vectors. How many support vectors are there compared to total samples?

**Medium:**
2. On the `make_moons` dataset, manually find a good C and gamma combination by iterating through values and plotting decision boundaries. Don't use GridSearchCV—develop intuition by visual inspection.

3. Load the `digits` dataset (8x8 images). Compare linear SVM vs. RBF SVM in terms of accuracy and number of support vectors. Why does one kernel work better than the other for image data?

**Hard:**
4. Write a function `plot_svm_decision(X, y, kernel, C, gamma)` that works for any 2D dataset and kernel type. It should plot the decision boundary, margins, and support vectors with a colorbar for the decision function values.

**Bonus:**
5. Apply SVM to regression using `SVR` with RBF kernel on the California housing dataset. Compare its performance to RandomForestRegressor. Hint: SVM for regression uses $\epsilon$-insensitive loss instead of hinge loss.

<details>
<summary><strong>💡 Exercise Solutions (Click to expand)</strong></summary>

### Exercise 1: Linear SVM on Iris (Binary)
```python
from sklearn.datasets import load_iris
from sklearn.svm import SVC

iris = load_iris()
X = iris.data[iris.target != 2][:, :2]  # Setosa vs Versicolor, first 2 features
y = iris.target[iris.target != 2]

svm = SVC(kernel='linear', C=1)
svm.fit(X, y)

print(f"Support vectors: {svm.n_support_} (total: {svm.support_.shape[0]})")
print(f"Total samples: {X.shape[0]}")
print(f"Ratio: {svm.support_.shape[0]/X.shape[0]:.2%}")

# Plot using the function from section 1
plot_svm_boundary(X, y, svm, 'Linear SVM on Iris (Setosa vs Versicolor)')
```

### Exercise 2: Manual C/Gamma Search
```python
X, y = make_moons(n_samples=300, noise=0.3, random_state=42)

C_values = [0.1, 1, 10]
gamma_values = [0.1, 1, 10]

fig, axes = plt.subplots(len(C_values), len(gamma_values), figsize=(12, 10))

for i, C in enumerate(C_values):
    for j, gamma in enumerate(gamma_values):
        svm = SVC(kernel='rbf', C=C, gamma=gamma)
        svm.fit(X, y)
        
        # Plot boundary (similar to section 4 code)
        # ... plotting code ...
        
        axes[i, j].set_title(f'C={C}, γ={gamma}')
```

### Exercise 3: Digits Dataset Comparison
```python
from sklearn.datasets import load_digits

digits = load_digits()
X_train, X_test, y_train, y_test = train_test_split(
    digits.data, digits.target, test_size=0.3, random_state=42
)

for kernel in ['linear', 'rbf']:
    svm = SVC(kernel=kernel, random_state=42)
    svm.fit(X_train, y_train)
    print(f"{kernel}: Accuracy={svm.score(X_test, y_test):.4f}, SVs={svm.support_.shape[0]}")
```

### Exercise 4: Generalized Plotting Function
```python
def plot_svm_decision(X, y, kernel='rbf', C=1.0, gamma='scale'):
    """Plot SVM decision boundary for 2D data."""
    svm = SVC(kernel=kernel, C=C, gamma=gamma)
    svm.fit(X, y)
    
    # Create mesh
    h = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    Z = svm.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, Z, levels=20, alpha=0.6, cmap='RdBu_r')
    plt.colorbar(label='Decision function value')
    plt.contour(xx, yy, Z, levels=[-1, 0, 1], colors='k', linestyles=['--', '-', '--'])
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu_r', edgecolors='k')
    plt.scatter(X[svm.support_, 0], X[svm.support_, 1], 
               s=200, facecolors='none', edgecolors='lime', linewidths=2)
    plt.title(f'SVM with {kernel} kernel (C={C}, γ={gamma})')
    plt.show()

# Test it
X, y = make_moons(noise=0.2, random_state=42)
plot_svm_decision(X, y, kernel='rbf', C=10, gamma=0.5)
```

### Exercise 5: SVR on California Housing
```python
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_squared_error

housing = fetch_california_housing()
X_train, X_test, y_train, y_test = train_test_split(
    housing.data, housing.target, test_size=0.2, random_state=42
)

# Scale features (essential for SVR)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# SVR with RBF
svr = SVR(kernel='rbf', C=100, gamma='scale')
svr.fit(X_train_s, y_train)
svr_pred = svr.predict(X_test_s)
svr_mse = mean_squared_error(y_test, svr_pred)

# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_mse = mean_squared_error(y_test, rf.predict(X_test))

print(f"SVR MSE: {svr_mse:.4f}")
print(f"RF MSE: {rf_mse:.4f}")
```
</details>

## 🎓 Summary – What You Learned Today

**Core Concepts:**
- SVM finds the **maximum margin** hyperplane, making it robust to overfitting in high dimensions
- **Support vectors** are the only data points that matter—everything else could be removed without changing the model
- **Soft margin** (parameter $C$) allows trading off between margin width and classification errors
- The **kernel trick** enables non-linear boundaries by computing similarities in high-dimensional space without explicit mapping

**Practical Skills:**
- Visualizing decision boundaries and margins for different kernels
- Tuning $C$ (regularization) and $\gamma$ (kernel width) using grid search
- Understanding OvO vs OvR strategies for multiclass problems
- **Always scaling features** before training SVM
- Building pipelines that combine preprocessing and model training

**When to Use SVM:**
- Medium-sized datasets ($n < 10000$) with high-dimensional features
- When decision boundary shape is unknown (RBF handles complexity automatically)
- When you need strong theoretical guarantees (max margin = good generalization)
- When interpretability of support vectors is valuable

**When NOT to Use SVM:**
- Very large datasets (use LinearSVC or other models instead)
- When probability estimates are needed (Platt scaling is expensive)
- When feature interpretability is required (kernel SVM is a black box)
- When training speed is critical

---